We compute the posterior probabilities $\theta_l$ and $\theta_r$ for the highest-ranked classifier in Table 6, namely Gradient Boosting. This focus is necessitated by the large number of possible comparisons, which makes a complete presentation of all results impractical. Gradient Boosting is selected as the representative classifier for analyzing posterior probabilities due to its consistently strong performance.

The results are shown in Figures 13 and 14. Figure 13 illustrates the performance of pipelines with and without resampling, while Figure 14 compares the performance of pipelines in which the order of the FS-R components is reversed.

# Generate the Tables in Figure 13

In [169]:
# Preparing the data for Figure 13

import os

d1 = {}

countries = [
    'polish',
    'russian',
    'taiwanese',
]

methods = [
    'f1_scores',
    'g_means',
    'roc_aucs',
]

clf = 'GradientBoostingClassifier'

blcs = [
    'RandomOverSampler',
    'RandomUnderSampler',
    'SMOTE',
]
fss = [
    #'Anova',
    't-Test',
    'Info',
    'GA',
    'PSO',
    'RandomForest',
    'LogisticRegressionWitL1Penalty',
]

for country in countries:
    for method in methods:
        for fs in fss:
            for blc in blcs:
                for flag in [True, False]:

                    fname0 = '_'.join([method, country])
                    fname1 = '_'.join([
                        fs,
                        blc,
                        clf,
                        str(flag),
                    ])
                    fname2 = '_'.join([
                        fs,
                        '---',
                        clf,
                        'True',
                    ])
    
                    fname = 'results_' + fname0 + '_' + fname1 + '___' + fname2 + '.jbl'

                    x = joblib.load(fr"C:\temp\{fname}")
                    thetas = x['thetas']
                    listOfValues = zip(*thetas)
                    tmpl = [np.mean(item) for item in listOfValues] #  P(Z < 0), P(Z = 0), P(Z > 0), Z = X - Y
                                                                    #  X is better, Same, Y is better
                    d1[(country, method, fs, blc, flag)] = tmpl

joblib.dump(d1, "d1.jbl")

['d1.jbl']

# Generate the Tables in Figure 14

In [147]:
# Preparing the data for Figure 14

import os

d2 = {}

#exist = []

countries = [
    'polish',
    'russian',
    'taiwanese',
]

methods = [
    'f1_scores',
    'g_means',
    'roc_aucs',
]

clf = 'GradientBoostingClassifier'

blcs = [
    'RandomOverSampler',
    'RandomUnderSampler',
    'SMOTE',
]
fss = [
    #'Anova',
    't-Test',
    'Info',
    'GA',
    'PSO',
    'RandomForest',
    'LogisticRegressionWitL1Penalty',
]

for country in countries:
    for method in methods:
        for fs in fss:
            for blc in blcs:


                # X is True, Y is False
                # X is bal+fs, Y is fs+bal
                fname0 = '_'.join([method, country])
                fname1 = '_'.join([
                    fs,
                    blc,
                    clf,
                    'True',
                ])
                fname2 = '_'.join([
                    fs,
                    blc,
                    clf,
                    'False',
                ])
    
                fname = 'results_' + fname0 + '_' + fname1 + '___' + fname2 + '.jbl'

                x = joblib.load(fr"C:\temp\{fname}")
                thetas = x['thetas']
                listOfValues = zip(*thetas)
                tmpl = [np.mean(item) for item in listOfValues] #  P(Z < 0), P(Z = 0), P(Z > 0), Z = X - Y
                d2[(country, method, fs, blc)] = tmpl

joblib.dump(d2, "d2.jbl")

['d2.jbl']

In [170]:
d = {
    'f1_scores': [],
    'g_means': [],
    'roc_aucs': [],
}

dd1 = {
    'polish': d.copy(),
    'taiwanese': d.copy(),
    'russian': d.copy(),
}

dd2 = {
    'polish': d.copy(),
    'taiwanese': d.copy(),
    'russian': d.copy(),
}

dd1

{'polish': {'f1_scores': [], 'g_means': [], 'roc_aucs': []},
 'taiwanese': {'f1_scores': [], 'g_means': [], 'roc_aucs': []},
 'russian': {'f1_scores': [], 'g_means': [], 'roc_aucs': []}}

In [171]:
for k1 in dd1:
    for k2 in dd1[k1]:
        x = {}
        
        for fs in fss:
            for blc in blcs:
                for flag in [True, False]:
                    #tmpl = d1[(country, method, fs, blc, flag)]
                    tmpl = d1[(k1, k2, fs, blc, flag)]
                    theta_up = tmpl[0]                          
                    theta_down = tmpl[-1]

                    # Z<0, Z>0 ; X<Y, X>Y
                    # X better ranked than Y
                    # Y better ranked than X
                    # Note that this is because, the lower the rank, the better
                    x[(fs, blc, flag)] = (theta_up, theta_down) 
                    
        dd1[k1][k2] = x

In [172]:
for k1 in dd2:
    for k2 in dd2[k1]:
        x = {}
        for fs in fss:
            for blc in blcs:
                #tmpl = d2[(country, method, fs, blc)]
                tmpl = d2[(k1, k2, fs, blc)]
                theta_up = tmpl[0]
                theta_down = tmpl[-1]

                x[(fs, blc)] = (theta_up, theta_down)
                    
        dd2[k1][k2] = x

In [173]:
joblib.dump(dd1, 'dd1.jbl')
joblib.dump(dd2, 'dd2.jbl')

['dd2.jbl']

In [8]:
from IPython.display import display, HTML
import pandas as pd

def display_dd1_table(d, decimals=3):
    """
    Display dd1['polish'] tables (f1_scores, g_means, roc_aucs) side by side
    with stacked values (first value on top, second below) and labels.

    Parameters
    ----------
    d : dict
        dd1['polish'], should contain keys 'f1_scores', 'g_means', 'roc_aucs',
        each mapping to {(fs, blc, flag): (v1, v2)}
        
    decimals : int
        Number of decimals to format the values
    """
    
    def format_cell(v1, v2):
        return f"{v1:.{decimals}f}\n{v2:.{decimals}f}"
    
    tables = {}

    # Build pivot tables
    for metric in ['f1_scores', 'g_means', 'roc_aucs']:
        metric_dict = d[metric]
        rows = []
        for (fs, blc, flag), (v1, v2) in metric_dict.items():
            rows.append({
                "Row": fs,
                "Col": (blc, flag),
                "Value": format_cell(float(v1), float(v2))
            })
        df = pd.DataFrame(rows)
        table = df.pivot(index="Row", columns="Col", values="Value")\
                  .sort_index(axis=0).sort_index(axis=1)
        tables[metric] = table

    # Convert to HTML with labels
    html_tables = []
    for metric, table in tables.items():
        html = f"<div><h3 style='text-align:center'>{metric}</h3>"
        html += table.style.set_table_styles(
            [{'selector': 'td', 'props': [('white-space', 'pre')]}]
        ).to_html()
        html += "</div>"
        html_tables.append(html)

    # Display all three tables side by side
    display(HTML(f"<div style='display:flex; gap:50px;'>{''.join(html_tables)}</div>"))

In [9]:
from IPython.display import display, HTML

def display_dd2_tables(d, decimals=3):
    """
    Display the three dd2 tables side by side with stacked values and labels.
    """
    def format_cell(v1, v2):
        return f"{v1:.{decimals}f}\n{v2:.{decimals}f}"

    tables = {}

    # Build tables with stacked values
    for metric, metric_dict in d.items():
        rows = []
        for (fs, blc), (v1, v2) in metric_dict.items():
            rows.append({
                "Row": fs,
                "Col": blc,
                "Value": format_cell(float(v1), float(v2))
            })
        df = pd.DataFrame(rows)
        table = df.pivot(index="Row", columns="Col", values="Value").sort_index(axis=0).sort_index(axis=1)
        tables[metric] = table

    # Convert each table to HTML with pre-formatted cells and add a label
    html_tables = []
    for metric in ['f1_scores', 'g_means', 'roc_aucs']:
        html = f"<div><h3 style='text-align:center'>{metric}</h3>"  # label
        html += tables[metric].style.set_table_styles(
            [{'selector': 'td', 'props': [('white-space', 'pre')]}]
        ).to_html()
        html += "</div>"
        html_tables.append(html)

    # Display tables side by side
    display(HTML(f"<div style='display:flex; gap:50px;'>{''.join(html_tables)}</div>"))

# Russian

In [1]:
import joblib

dd1 = joblib.load('dd1.jbl')
dd2 = joblib.load('dd2.jbl')

In [6]:
display_dd1_table(dd1['russian'])

Col,"('RandomOverSampler', False)","('RandomOverSampler', True)","('RandomUnderSampler', False)","('RandomUnderSampler', True)","('SMOTE', False)","('SMOTE', True)"
Row,,,,,,
GA,0.700 0.298,0.726 0.272,0.347 0.652,0.351 0.648,0.582 0.417,0.700 0.299
Info,0.646 0.332,0.715 0.283,0.427 0.549,0.592 0.405,0.631 0.349,0.719 0.278
LogisticRegressionWitL1Penalty,0.617 0.381,0.624 0.373,0.278 0.719,0.237 0.762,0.424 0.575,0.657 0.341
PSO,0.615 0.384,0.650 0.347,0.314 0.685,0.289 0.711,0.497 0.500,0.660 0.336
RandomForest,0.702 0.297,0.683 0.316,0.485 0.514,0.468 0.531,0.615 0.382,0.791 0.207
t-Test,0.678 0.269,0.708 0.273,0.430 0.527,0.483 0.502,0.522 0.434,0.675 0.292
Col,"('RandomOverSampler', False)","('RandomOverSampler', True)","('RandomUnderSampler', False)","('RandomUnderSampler', True)","('SMOTE', False)","('SMOTE', True)"
Row,,,,,,
GA,1.000 0.000,1.000 0.000,1.000 0.000,0.999 0.001,0.992 0.008,0.999 0.001


In [7]:
display_dd2_tables(dd2['russian'])

Col,RandomOverSampler,RandomUnderSampler,SMOTE
Row,,,
GA,0.492 0.478,0.463 0.520,0.616 0.381
Info,0.576 0.412,0.652 0.323,0.586 0.411
LogisticRegressionWitL1Penalty,0.490 0.487,0.362 0.619,0.769 0.228
PSO,0.477 0.499,0.428 0.552,0.763 0.234
RandomForest,0.397 0.445,0.411 0.432,0.812 0.185
t-Test,0.431 0.419,0.485 0.457,0.693 0.274
Col,RandomOverSampler,RandomUnderSampler,SMOTE
Row,,,
GA,0.476 0.494,0.438 0.546,0.284 0.714


# Taiwanese

In [10]:
import joblib

dd1 = joblib.load('dd1.jbl')
dd2 = joblib.load('dd2.jbl')

display_dd1_table(dd1['taiwanese'])
display_dd2_tables(dd2['taiwanese'])

Col,"('RandomOverSampler', False)","('RandomOverSampler', True)","('RandomUnderSampler', False)","('RandomUnderSampler', True)","('SMOTE', False)","('SMOTE', True)"
Row,,,,,,
GA,0.673 0.326,0.732 0.265,0.197 0.801,0.223 0.775,0.664 0.335,0.817 0.182
Info,0.689 0.310,0.716 0.283,0.268 0.731,0.238 0.761,0.672 0.327,0.749 0.250
LogisticRegressionWitL1Penalty,0.731 0.268,0.765 0.233,0.206 0.791,0.174 0.824,0.766 0.231,0.829 0.168
PSO,0.715 0.283,0.760 0.238,0.216 0.783,0.214 0.785,0.657 0.342,0.841 0.158
RandomForest,0.708 0.291,0.698 0.301,0.294 0.703,0.260 0.737,0.653 0.345,0.749 0.247
t-Test,0.771 0.228,0.757 0.242,0.260 0.739,0.299 0.699,0.753 0.245,0.776 0.223
Col,"('RandomOverSampler', False)","('RandomOverSampler', True)","('RandomUnderSampler', False)","('RandomUnderSampler', True)","('SMOTE', False)","('SMOTE', True)"
Row,,,,,,
GA,1.000 0.000,1.000 0.000,1.000 0.000,1.000 0.000,1.000 0.000,1.000 0.000


Col,RandomOverSampler,RandomUnderSampler,SMOTE
Row,,,
GA,0.563 0.424,0.580 0.415,0.744 0.252
Info,0.580 0.411,0.383 0.612,0.739 0.257
LogisticRegressionWitL1Penalty,0.517 0.475,0.454 0.540,0.652 0.343
PSO,0.542 0.451,0.534 0.460,0.796 0.202
RandomForest,0.506 0.481,0.400 0.593,0.768 0.227
t-Test,0.488 0.502,0.562 0.432,0.623 0.370
Col,RandomOverSampler,RandomUnderSampler,SMOTE
Row,,,
GA,0.515 0.476,0.515 0.479,0.641 0.355


# Polish

In [11]:
import joblib

dd1 = joblib.load('dd1.jbl')
dd2 = joblib.load('dd2.jbl')

display_dd1_table(dd1['polish'])
display_dd2_tables(dd2['polish'])

Col,"('RandomOverSampler', False)","('RandomOverSampler', True)","('RandomUnderSampler', False)","('RandomUnderSampler', True)","('SMOTE', False)","('SMOTE', True)"
Row,,,,,,
GA,0.662 0.334,0.696 0.301,0.153 0.845,0.234 0.765,0.543 0.453,0.612 0.383
Info,0.056 0.939,0.098 0.897,0.001 0.999,0.001 0.999,0.036 0.962,0.017 0.982
LogisticRegressionWitL1Penalty,0.820 0.177,0.846 0.152,0.341 0.656,0.629 0.370,0.683 0.315,0.774 0.223
PSO,0.652 0.344,0.694 0.303,0.240 0.758,0.298 0.700,0.563 0.435,0.675 0.323
RandomForest,0.054 0.937,0.033 0.961,0.000 0.999,0.000 1.000,0.024 0.975,0.004 0.996
t-Test,0.684 0.314,0.716 0.282,0.157 0.842,0.457 0.540,0.520 0.478,0.645 0.352
Col,"('RandomOverSampler', False)","('RandomOverSampler', True)","('RandomUnderSampler', False)","('RandomUnderSampler', True)","('SMOTE', False)","('SMOTE', True)"
Row,,,,,,
GA,1.000 0.000,0.999 0.001,1.000 0.000,1.000 0.000,1.000 0.000,0.998 0.001


Col,RandomOverSampler,RandomUnderSampler,SMOTE
Row,,,
GA,0.615 0.366,0.540 0.445,0.632 0.361
Info,0.578 0.385,0.441 0.520,0.417 0.571
LogisticRegressionWitL1Penalty,0.735 0.249,0.899 0.086,0.675 0.322
PSO,0.644 0.342,0.531 0.455,0.725 0.269
RandomForest,0.323 0.629,0.173 0.803,0.512 0.470
t-Test,0.582 0.389,0.864 0.123,0.796 0.201
Col,RandomOverSampler,RandomUnderSampler,SMOTE
Row,,,
GA,0.567 0.414,0.562 0.421,0.542 0.454
